# Xbar-S Analysis

Xbar-S charts are the gold standard for monitoring processes with subgrouped data. They're ideal when you have:

- Multiple measurements per time period
- Rational subgroups (factors like machines, operators, batches)
- Need to monitor both location (mean) and spread (variation)

## What You'll Learn

1. Create Xbar and S charts from replicated data
2. Understand how design states affect variance estimation
3. Compare factor levels using control charts
4. Access VAS residuals for deeper analysis

## Setup

In [1]:
import numpy as np
import pandas as pd
from processbehavior import ProcessBehavior

## Create Replicated Data

We'll simulate a filling machine with:
- 3 operators (A, B, C)
- 8 time periods
- 4 replicate measurements per operator per time period

This creates **design state 1 (full replication)** — the most powerful design on Bishop's 1–6 reference scale.

In [2]:
np.random.seed(42)

operators = ['A', 'B', 'C']
n_times = 8
n_reps = 4

data = []
for t in range(n_times):
    for op in operators:
        # Each operator has a slightly different mean
        op_effect = {'A': 0, 'B': 2, 'C': -1}[op]
        
        # Add time trend (process drift)
        time_effect = t * 0.3
        
        for rep in range(n_reps):
            # Add special cause for Operator B at time 6
            special = 8 if (op == 'B' and t == 6) else 0
            
            value = 100 + op_effect + time_effect + special + np.random.normal(0, 1.5)
            data.append({
                'time': t + 1,
                'operator': op,
                'weight': round(value, 2)
            })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} observations")
print(f"Structure: {len(operators)} operators x {n_times} times x {n_reps} reps")
df.head(12)

Dataset: 96 observations
Structure: 3 operators x 8 times x 4 reps


,time,operator,weight
0,1,A,100.75
1,1,A,99.79
2,1,A,100.97
3,1,A,102.28
4,1,B,101.65
5,1,B,101.65
6,1,B,104.37
7,1,B,103.15
8,1,C,98.30
9,1,C,99.81


## Formulate the Study

In [3]:
pb = ProcessBehavior(df)

study = pb.formulate(
    response=pb.cols.weight,
    factors=[pb.cols.operator],
    time=pb.cols.time
)

print(f"ADS: {study.analytical_design_state.sds} ({study.ads_reason})")
print(f"Description: {study.ads_description}")
print(f"\nValid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")
print(f"Residual charts: {study.residual_charts}")

ADS: 1 (full_replication)
Description: Full replication (all cells n≥2)

Valid charts: ['Histogram', 'Xbar', 'S', 'X', 'mR']
Recommended: Xbar
Residual charts: [('Xbar', 'R1'), ('X', 'R1'), ('S', 'R2'), ('X', 'R2'), ('Xbar', 'R3'), ('S', 'R3'), ('Xbar', 'R4'), ('S', 'R4'), ('Xbar', 'R5'), ('S', 'R5'), ('Xbar', 'R6'), ('S', 'R6')]


## Understanding design state 1

**Design state 1 (full replication)** is the most powerful because:

1. Every (operator, time) cell has multiple observations
2. Within-cell variance can be estimated exactly
3. All stored VAS residuals (R1–R5) are computed, and the per-request R6 is available
4. Interactions can be detected

The formula for control limits uses the pooled within-cell standard deviation.

## Execute Xbar-S Charts Analysis

In [4]:
# companion=True computes the S chart alongside the recommended Xbar —
# execute() alone returns only the recommended chart.
result = study.execute(companion=True)

print(f"Charts created: {result.all_charts}")
print(f"Has residuals: {result.has_residuals}")

Charts created: ['Xbar', 'S']
Has residuals: True


## View Chart Data

In [5]:
# Xbar chart shows subgroup means
xbar_data = result.get_chart('Xbar')
print("Xbar Chart Data (subgroup means):")
xbar_data.head(10)

Xbar Chart Data (subgroup means):


,group,xbar,center,lpl,upl,beyond_limits
0,A_1,100.948,101.549,99.448,103.651,0
1,A_2,98.815,101.549,99.448,103.651,-1
2,A_3,100.145,101.549,99.448,103.651,0
3,A_4,99.820,101.549,99.448,103.651,0
4,A_5,100.648,101.549,99.448,103.651,0
5,A_6,100.388,101.549,99.448,103.651,0
6,A_7,101.700,101.549,99.448,103.651,0
7,A_8,102.075,101.549,99.448,103.651,0
8,B_1,102.705,101.549,99.448,103.651,0
9,B_2,101.168,101.549,99.448,103.651,0


In [6]:
# S chart shows subgroup standard deviations
s_data = result.get_chart('S')
print("\nS Chart Data (subgroup std devs):")
s_data.head(10)


S Chart Data (subgroup std devs):


,group,s,center,lpl,upl,beyond_limits
0,A_1,1.025,1.29,0.0,2.924,0
1,A_2,1.523,1.29,0.0,2.924,0
2,A_3,1.030,1.29,0.0,2.924,0
3,A_4,1.646,1.29,0.0,2.924,0
4,A_5,1.483,1.29,0.0,2.924,0
5,A_6,0.732,1.29,0.0,2.924,0
6,A_7,2.735,1.29,0.0,2.924,0
7,A_8,1.175,1.29,0.0,2.924,0
8,B_1,1.316,1.29,0.0,2.924,0
9,B_2,1.117,1.29,0.0,2.924,0


In [7]:
# Statistics for both charts
print("Xbar Statistics:")
display(result.get_statistics('Xbar'))

print("\nS Statistics:")
display(result.get_statistics('S'))

Xbar Statistics:


{'center': np.float64(101.549),
 'N': np.int64(4),
 'upl': np.float64(103.651),
 'lpl': np.float64(99.448)}


S Statistics:


{'center': np.float64(1.29),
 'N': np.int64(4),
 'upl': np.float64(2.924),
 'lpl': np.float64(0.0)}

## Visualize Xbar Chart

In [8]:
fig = result.plot(
    chart='Xbar',
    show_zones=True,
    highlight_signals=True,
    show_stats=True
)
fig.show()

## Visualize S Chart

In [9]:
fig = result.plot(
    chart='S',
    show_zones=True,
    highlight_signals=True
)
fig.show()

## Understanding Xbar-S Charts

### The Xbar Chart

- Plots the **mean** of each subgroup
- Centerline: Grand mean of all observations
- Limits based on within-subgroup variation
- Detects **shifts in process level**

### The S Chart

- Plots the **standard deviation** of each subgroup
- Centerline: Pooled within-subgroup standard deviation
- Limits based on chi-square distribution
- Detects **changes in process variation**

### Reading Order

1. **First check the S chart** - Variation must be stable
2. **Then interpret the Xbar chart** - Only valid if S is stable
3. Points on Xbar beyond limits → investigate the specific subgroup

## Signal Detection for Xbar-S

For Xbar and S charts (categorical comparisons), only **Rule 1** applies - points beyond the control limits.

In [10]:
# Detect signals on Xbar
signals = result.detect_signals(chart='Xbar')

print(f"Xbar signals: {signals.count}")
if signals.has_signals:
    print("\nViolations:")
    display(signals.violations)

Xbar signals: 9

Violations:


,obs_id,rule_name,rule_number,description,value,center,upl,lpl
0,1,rule_1,1,Point beyond control limits,98.815,101.549,103.651,99.448
1,12,rule_1,1,Point beyond control limits,103.912,101.549,103.651,99.448
2,13,rule_1,1,Point beyond control limits,104.662,101.549,103.651,99.448
3,14,rule_1,1,Point beyond control limits,111.010,101.549,103.651,99.448
4,15,rule_1,1,Point beyond control limits,104.495,101.549,103.651,99.448
5,16,rule_1,1,Point beyond control limits,98.678,101.549,103.651,99.448
6,17,rule_1,1,Point beyond control limits,99.255,101.549,103.651,99.448
7,18,rule_1,1,Point beyond control limits,99.048,101.549,103.651,99.448
8,19,rule_1,1,Point beyond control limits,99.300,101.549,103.651,99.448


In [11]:
# Detect signals on Sbar
signals_s = result.detect_signals(chart='S')

print(f"S chart signals: {signals_s.count}")

S chart signals: 0


## Accessing VAS Residuals

With full replication, the stored residuals R1–R5 are computed at `formulate()` time and live on the result. (R6 is request-scoped — computed per `execute(value='R6', by=...)` call — so it is deliberately not in this frame.)

In [12]:
# View the computed residuals
residuals = result.residuals
print("VAS Residuals:")
residuals.head(10)

VAS Residuals:


,R1,R2,R3,R4,R5
0,-0.799479,-0.1975,0.955625,-0.970312,-1.179792
1,-1.759479,-1.1575,-0.004375,-1.930312,-2.139792
2,-0.579479,0.0225,1.175625,-0.750312,-0.959792
3,0.730521,1.3325,2.485625,0.559688,0.350208
12,-0.889479,1.8450,1.896458,0.041354,0.862708
13,-4.119479,-1.3850,-1.333542,-3.188646,-2.367292
14,-3.839479,-1.1050,-1.053542,-2.908646,-2.087292
15,-2.089479,0.6450,0.696458,-1.158646,-0.337292
24,-1.769479,-0.3650,0.119792,-1.271979,-1.347292
25,-0.779479,0.6250,1.109792,-0.281979,-0.357292


In [13]:
# Analyze time effects using R4 residuals on S chart
result_r4 = study.execute(chart='S', value='R4')

# View the chart data
print("R4 Residual on S Chart (Time Effects):")
print(f"Charts: {result_r4.all_charts}")
result_r4.get_chart('S').head()

R4 Residual on S Chart (Time Effects):
Charts: ['S']


,group,s,center,lpl,upl,beyond_limits
0,A_1,1.025,1.29,0.0,2.924,0
1,A_2,1.523,1.29,0.0,2.924,0
2,A_3,1.030,1.29,0.0,2.924,0
3,A_4,1.646,1.29,0.0,2.924,0
4,A_5,1.483,1.29,0.0,2.924,0


In [14]:
# Analyze factor (operator) effects using R5 residuals on S chart
result_r5 = study.execute(chart='S', value='R5')

# View the chart data
print("R5 Residual on S Chart (Operator Effects):")
print(f"Charts: {result_r5.all_charts}")
result_r5.get_chart('S').head()

R5 Residual on S Chart (Operator Effects):
Charts: ['S']


,group,s,center,lpl,upl,beyond_limits
0,A_1,1.025,1.29,0.0,2.924,0
1,A_2,1.523,1.29,0.0,2.924,0
2,A_3,1.030,1.29,0.0,2.924,0
3,A_4,1.646,1.29,0.0,2.924,0
4,A_5,1.483,1.29,0.0,2.924,0


## Chart Table Summary

Get a compact summary table for reporting:

In [15]:
# Summary table with subgroup info, values, and limits
table = result.chart_table('Xbar')
table

,subgroup,n,value,center,lpl,upl,signal
1,A_1,4,100.948,101.549,99.448,103.651,
2,A_2,4,98.815,101.549,99.448,103.651,↓
3,A_3,4,100.145,101.549,99.448,103.651,
4,A_4,4,99.820,101.549,99.448,103.651,
5,A_5,4,100.648,101.549,99.448,103.651,
6,A_6,4,100.388,101.549,99.448,103.651,
7,A_7,4,101.700,101.549,99.448,103.651,
8,A_8,4,102.075,101.549,99.448,103.651,
9,B_1,4,102.705,101.549,99.448,103.651,
10,B_2,4,101.168,101.549,99.448,103.651,


## Summary

In this tutorial, you learned:

- Xbar-S charts require subgrouped data (n >= 2 per cell)
- Design state 1 (full replication) provides the most analytical power
- The S chart monitors variation; the Xbar chart monitors level
- Only Rule 1 applies to Xbar-S charts
- VAS residuals enable deeper root cause analysis

## Next Steps

- [Stratified Analysis](stratified-analysis.ipynb) - Separate charts per factor level
- [VAS Residuals](../user-guide/residuals.md) - Deep dive into VAS residuals
- [Signal Detection](signal-detection.ipynb) - All Western Electric rules